# Análise da Rotatividade de Colaboradores

## Uma Abordagem Orientada a Dados para Compreender a Saída de Funcionários

**Objetivo do Projeto:** Identificar os fatores-chave que impulsionam a rotatividade de colaboradores e propor recomendações baseadas em evidências para reduzir a saída voluntária.

**Dataset:** 1.470 colaboradores | 35 variáveis | Dados de Recursos Humanos

**Métricas Principais Analisadas:**
- Remuneração e Progressão de Carreira
- Impacto de Horas Extras na Saída
- Perfil Demográfico e Grupos de Risco
- Correlação com Variáveis de RH


## 1. Configuração e Carregamento de Dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# Configurar visualizações
%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

In [ ]:
# Carregar dataset
df = pd.read_csv("Employee_Attrition.csv")
print(f"Dimensão do dataset: {df.shape}")
print(f"Variáveis: {df.columns.tolist()}")
df.head()

## 2. Avaliação da Qualidade dos Dados

Verificar integridade dos dados, valores ausentes e tipos de dados antes da análise.

In [ ]:
# Verificar valores em falta
valores_faltantes = df.isna().sum()
print("Valores em Falta:")
print(valores_faltantes[valores_faltantes > 0] if valores_faltantes.sum() > 0 else "Nenhum valor em falta encontrado")

# Tipos de dados
print("\nTipos de Dados:")
print(df.dtypes)

# Estatísticas descritivas
print("\nEstatísticas Descritivas:")
df.describe().round(2)

## 3. Preparação de Dados

Criar variáveis derivadas para melhor análise.

In [ ]:
# Criar cópia de trabalho
df_analise = df.copy()

# Converter Attrition (Sim/Não) para binário para cálculos
df_analise['Attrition_Binario'] = (df_analise['Attrition'] == 'Yes').astype(int)

# Converter OverTime (Sim/Não) para binário
df_analise['HorasExtras_Binario'] = (df_analise['OverTime'] == 'Yes').astype(int)

print("✓ Preparação de dados concluída.")
print(f"Dimensão: {df_analise.shape}")

## 4. Visão Geral Organizacional

Resumo de alto nível da força de trabalho e linha de base de rotatividade.

In [ ]:
# Taxa de rotatividade geral
total_colaboradores = len(df_analise)
saidas = (df_analise['Attrition'] == 'Yes').sum()
retidos = (df_analise['Attrition'] == 'No').sum()
taxa_rotatividade = (saidas / total_colaboradores) * 100

print(f"Total de Colaboradores: {total_colaboradores}")
print(f"Colaboradores que Saíram: {saidas} ({taxa_rotatividade:.2f}%)")
print(f"Colaboradores Retidos: {retidos} ({100-taxa_rotatividade:.2f}%)")

In [ ]:
# Gráfico de pizza de rotatividade
contagem_rotatividade = df_analise['Attrition'].value_counts()

fig, ax = plt.subplots(figsize=(8, 6))
cores = ['#2ecc71', '#e74c3c']
fatias, textos, autotextos = ax.pie(contagem_rotatividade, 
                                      labels=['Sem Saída', 'Saída'],
                                      autopct='%1.1f%%',
                                      colors=cores,
                                      startangle=90)
ax.set_title('Taxa de Rotatividade Global', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Análise de Remuneração

**Descoberta Principal:** Colaboradores que saem ganham 29,4% menos que os que ficam.

Remuneração baixa é um impulsionador significativo de rotatividade.

In [ ]:
# Análise de salário por rotatividade
salario_por_rotatividade = df_analise.groupby('Attrition')['MonthlyIncome'].mean()

fig, ax = plt.subplots(figsize=(10, 6))
barras = ax.bar(salario_por_rotatividade.index, salario_por_rotatividade.values, 
                color=['#3498db', '#e74c3c'], width=0.6, edgecolor='black')
ax.set_ylabel('Rendimento Médio Mensal (USD)', fontsize=11)
ax.set_xlabel('Status de Rotatividade', fontsize=11)
ax.set_title('Salário Médio por Status de Saída', fontsize=13, fontweight='bold')
ax.set_xticklabels(['Sem Saída', 'Saída'])
ax.set_ylim(0, 8000)

# Adicionar labels de valores nas barras
for barra in barras:
    altura = barra.get_height()
    ax.text(barra.get_x() + barra.get_width()/2., altura,
            f'USD {altura:,.0f}',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

# Calcular diferença percentual
diff_salario = ((salario_por_rotatividade['No'] - salario_por_rotatividade['Yes']) / salario_por_rotatividade['No']) * 100
print(f"Diferença salarial: {diff_salario:.1f}%")

In [ ]:
# Salário por Cargo
salario_por_cargo = df_analise.groupby('JobRole')['MonthlyIncome'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(11, 6))
barras = ax.barh(salario_por_cargo.index, salario_por_cargo.values, color='#3498db', edgecolor='black')
ax.set_xlabel('Rendimento Médio Mensal (USD)', fontsize=11)
ax.set_title('Salário Médio por Cargo', fontsize=13, fontweight='bold')
ax.invert_yaxis()

# Adicionar labels de valores
for i, barra in enumerate(barras):
    largura = barra.get_width()
    ax.text(largura, barra.get_y() + barra.get_height()/2.,
            f'USD {largura:,.0f}',
            ha='left', va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Análise de Carga de Trabalho e Horas Extras

**DESCOBERTA CRÍTICA:** Horas extras TRIPLICAM a taxa de rotatividade (10,4% → 30,5%).

Este é o factor mais forte e isolado que afecta a retenção de colaboradores.

In [ ]:
# Impacto de horas extras na rotatividade
rotatividade_horas_extras = df_analise.groupby('OverTime')['Attrition_Binario'].agg(['sum', 'count'])
rotatividade_horas_extras['taxa_%'] = (rotatividade_horas_extras['sum'] / rotatividade_horas_extras['count'] * 100).round(1)

print("Taxa de Rotatividade por Status de Horas Extras:")
print(rotatividade_horas_extras)

fig, ax = plt.subplots(figsize=(9, 6))
x_pos = np.arange(len(rotatividade_horas_extras))
barras = ax.bar(x_pos, rotatividade_horas_extras['taxa_%'], 
                color=['#2ecc71', '#e74c3c'], width=0.6, edgecolor='black')
ax.set_ylabel('Taxa de Rotatividade (%)', fontsize=11)
ax.set_xlabel('Status de Horas Extras', fontsize=11)
ax.set_title('Impacto de Horas Extras na Taxa de Rotatividade', fontsize=13, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels(['Sem Horas Extras', 'Com Horas Extras'])
ax.set_ylim(0, 35)

# Adicionar labels de valores
for barra in barras:
    altura = barra.get_height()
    ax.text(barra.get_x() + barra.get_width()/2., altura,
            f'{altura:.1f}%',
            ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## 7. Análise de Progressão de Carreira

**Descoberta:** 35% da rotatividade ocorre nos primeiros anos na empresa.

Colaboradores em fase inicial de carreira lutam com falta de avanço e crescimento.

In [ ]:
# Análise de tempo na empresa
tempo_por_rotatividade = df_analise.groupby('Attrition')['YearsAtCompany'].mean()

print("Tempo Médio na Empresa por Status de Rotatividade:")
print(f"Retidos: {tempo_por_rotatividade['No']:.1f} anos")
print(f"Saídas: {tempo_por_rotatividade['Yes']:.1f} anos")

# Anos desde última promoção
promocao_por_rotatividade = df_analise.groupby('Attrition')['YearsSinceLastPromotion'].mean()
print(f"\nTempo Médio Desde Última Promoção:")
print(f"Retidos: {promocao_por_rotatividade['No']:.1f} anos")
print(f"Saídas: {promocao_por_rotatividade['Yes']:.1f} anos")

In [ ]:
# Distribuição de tempo na empresa por rotatividade
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Colaboradores retidos
df_analise[df_analise['Attrition'] == 'No']['YearsAtCompany'].hist(
    bins=30, ax=axes[0], color='#2ecc71', edgecolor='black', alpha=0.7)
axes[0].set_title('Distribuição de Tempo - Colaboradores Retidos', fontweight='bold')
axes[0].set_xlabel('Anos na Empresa')
axes[0].set_ylabel('Contagem')

# Colaboradores que saíram
df_analise[df_analise['Attrition'] == 'Yes']['YearsAtCompany'].hist(
    bins=30, ax=axes[1], color='#e74c3c', edgecolor='black', alpha=0.7)
axes[1].set_title('Distribuição de Tempo - Colaboradores que Saíram', fontweight='bold')
axes[1].set_xlabel('Anos na Empresa')
axes[1].set_ylabel('Contagem')

plt.tight_layout()
plt.show()

## 8. Perfil Demográfico e Grupos de Risco

**Grupos de Risco Principal:**
- Idades 20-25: 35,8% de rotatividade (fase de exploração)
- Idades 26-35: 19,1% de rotatividade (procura de crescimento)
- Idade 36+: <13% de rotatividade (fase estabilizada)

In [ ]:
# Criar grupos etários
df_analise['GrupoIdade'] = pd.cut(df_analise['Age'], 
                                   bins=[0, 25, 35, 45, 100],
                                   labels=['20-25', '26-35', '36-45', '45+'])

rotatividade_idade = df_analise.groupby('GrupoIdade')['Attrition_Binario'].agg(['sum', 'count'])
rotatividade_idade['taxa_%'] = (rotatividade_idade['sum'] / rotatividade_idade['count'] * 100).round(1)

print("Taxa de Rotatividade por Grupo Etário:")
print(rotatividade_idade)

fig, ax = plt.subplots(figsize=(10, 6))
cores_idade = ['#e74c3c', '#f39c12', '#f1c40f', '#2ecc71']
barras = ax.bar(rotatividade_idade.index.astype(str), rotatividade_idade['taxa_%'], 
                color=cores_idade, width=0.6, edgecolor='black')
ax.set_ylabel('Taxa de Rotatividade (%)', fontsize=11)
ax.set_xlabel('Grupo Etário', fontsize=11)
ax.set_title('Taxa de Rotatividade por Grupo Etário (Ciclo de Carreira)', fontsize=13, fontweight='bold')
ax.set_ylim(0, 40)

# Adicionar labels de valores
for barra in barras:
    altura = barra.get_height()
    ax.text(barra.get_x() + barra.get_width()/2., altura,
            f'{altura:.1f}%',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

## 9. Análise de Correlações

Identificar quais factores têm a relação mais forte com a rotatividade.

In [ ]:
# Selecionar colunas numéricas para correlação
colunas_numericas = df_analise.select_dtypes(include=[np.number]).columns
matriz_correlacao = df_analise[colunas_numericas].corr()

# Obter correlações com Attrition
correlacao_attrition = matriz_correlacao['Attrition_Binario'].sort_values(ascending=False)

print("Factores Mais Correlacionados com Rotatividade:")
print(correlacao_attrition.head(10))

# Visualizar heatmap de correlação
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(matriz_correlacao, annot=False, cmap='coolwarm', center=0, 
            square=True, ax=ax, cbar_kws={'label': 'Coeficiente de Correlação'})
ax.set_title('Matriz de Correlação - Todas as Variáveis Numéricas', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Descobertas-Chave e Recomendações

### Descobertas Críticas

**1. Horas Extras é o Impulsionador #1** (30,5% rotatividade vs 10,4% baseline)
   - Impacto desproporcional nas equipas de Investigação e Vendas
   - **Recomendação:** Implementar limites rigorosos de horas extras e revisão de compensação

**2. Disparidade de Remuneração** (diferença salarial de 29,4%)
   - Cargos entry-level particularmente afectados
   - Sales Representatives, Lab Technicians, HR mais em risco
   - **Recomendação:** Benchmarking salarial para funções de alto risco

**3. Instabilidade de Carreira Inicial** (35% de rotatividade nos primeiros 2 anos)
   - Colaboradores jovens (20-25) têm 35,8% de rotatividade
   - Estagnação de carreira após 2,2 anos na mesma função
   - **Recomendação:** Planos de progressão de carreira + programas de mentoria

### Acções Recomendadas

**Imediato (0-3 meses):**
✓ Estabelecer limites de horas extras (máx. 10-15% de horas/mês)  
✓ Implementar revisão obrigatória de compensação de horas extras  
✓ Identificar indivíduos de alto risco para contacto de retenção

**Curto Prazo (3-6 meses):**
✓ Conduzir benchmarking salarial para funções com >20% rotatividade  
✓ Estabelecer trajectórias claras de progressão de carreira  
✓ Criar estratégias de retenção específicas por cargo

**Longo Prazo (6-12 meses):**
✓ Lançar programas de mentoria para colaboradores em fase inicial  
✓ Implementar ciclos de promoção regulares (a cada 2,0-2,5 anos)  
✓ Desenvolver caminhos de aprendizagem especializados para funções de alto risco


## 11. Tabela de Resumo

In [ ]:
# Criar tabela de estatísticas resumidas
stats_resumo = {
    'Métrica': [
        'Total de Colaboradores',
        'Colaboradores que Saíram',
        'Taxa de Rotatividade Global',
        'Salário Médio (Retidos)',
        'Salário Médio (Saídas)',
        'Diferença Salarial',
        'Taxa Rotatividade (Sem Horas Extras)',
        'Taxa Rotatividade (Com Horas Extras)',
        'Multiplicador de Horas Extras',
        'Grupo de Risco Máximo',
        'Cargo com Maior Rotatividade'
    ],
    'Valor': [
        f"{len(df_analise)}",
        f"{saidas}",
        f"{taxa_rotatividade:.2f}%",
        f"USD {salario_por_rotatividade['No']:,.0f}",
        f"USD {salario_por_rotatividade['Yes']:,.0f}",
        f"{diff_salario:.1f}%",
        f"{rotatividade_horas_extras.loc['No', 'taxa_%']:.1f}%",
        f"{rotatividade_horas_extras.loc['Yes', 'taxa_%']:.1f}%",
        f"{rotatividade_horas_extras.loc['Yes', 'taxa_%'] / rotatividade_horas_extras.loc['No', 'taxa_%']:.1f}x",
        "20-25 anos (35,8%)",
        "Sales Representative (39,75%)"
    ]
}

df_resumo = pd.DataFrame(stats_resumo)
display(df_resumo)